# AlgoVault MCP + CrewAI

**Composite BUY/SELL/HOLD trade calls across 5 crypto perp venues. Verified track record, Merkle-anchored on Base L2.**

[AlgoVault](https://algovault.com) exposes a remote MCP server that returns one composite verdict per call — direction, confidence, market regime, and cross-venue funding context — instead of raw indicators you have to vote-count yourself.

This notebook connects CrewAI (a multi-agent orchestration framework) to that server with `MCPServerAdapter`, loads the AlgoVault tools, and reads a live trade call. The direct tool-call path needs no LLM key.

### What is specific to AlgoVault

- **Composite verdict, not raw signals.** One JSON response (`call`, `confidence`, `indicators`, `regime`) replaces multi-indicator vote-counting.
- **Cross-venue.** Signal is fused across 5 crypto perp venues — Binance, Bybit, OKX, Bitget, Hyperliquid — not derivable from any single-venue API.
- **On-chain verifiable.** Each signal is Merkle-anchored to Base L2 under [ERC-8004 agentId `44544`](https://basescan.org/token/0x8004A169FB4a3325136EB29fA0ceB6D2e539a432?a=44544). The track record is public, not a claim.

Track record as of 2026-06-04: a 91.6% PFE win rate over 150,695+ calls, 55+ on-chain batches, 763 assets covered. Numbers refresh live — verify them yourself at [algovault.com/track-record](https://algovault.com/track-record).

> Informational analytics, not investment advice. Past performance does not guarantee future results.

## Prerequisites

- **Python 3.10+** (`python3 --version`). CrewAI requires it.
- **`crewai` + `crewai-tools[mcp]`**, pinned in the install cell below. The two packages are released in lockstep, so keep their versions matched.
- **No AlgoVault API key needed** for the direct tool-call path in this notebook. AlgoVault's free tier covers a daily quota per IP — enough for local exploration.
- **(Optional) an LLM provider key** (`OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, …) only for the final `Crew` cell, which is guarded and skipped if no key is set.

In [1]:
%pip install -U --quiet "crewai==1.15.1" "crewai-tools[mcp]==1.15.1"

Note: you may need to restart the kernel to use updated packages.


## 1. Connect and load the AlgoVault tools

`MCPServerAdapter` opens a streamable-HTTP connection to the AlgoVault MCP endpoint and exposes each MCP tool as a CrewAI tool. Use it as a context manager so the connection is closed cleanly.

Note the transport value: CrewAI's adapter expects `"streamable-http"` with a **hyphen**.

The four core trade tools are listed below.

In [2]:
from crewai_tools import MCPServerAdapter

ALGOVAULT_MCP_URL = "https://api.algovault.com/mcp"

server_params = {
    "url": ALGOVAULT_MCP_URL,
    "transport": "streamable-http",  # hyphen, not underscore
}

CORE_TOOLS = [
    "get_trade_call",
    "get_trade_signal",
    "scan_funding_arb",
    "get_market_regime",
]

with MCPServerAdapter(server_params) as algovault_tools:
    available = {t.name for t in algovault_tools}

print("Loaded AlgoVault trade tools:")
for name in CORE_TOOLS:
    mark = "ok" if name in available else "missing"
    print(f"  - {name} ({mark})")

Loaded AlgoVault trade tools:
  - get_trade_call (ok)
  - get_trade_signal (ok)
  - scan_funding_arb (ok)
  - get_market_regime (ok)


## 2. Read a live trade call (no LLM key)

Invoke `get_trade_call` directly through the CrewAI tool object. The `coin` argument is the only required field; the rest below pin an explicit venue, timeframe, and asset class.

The parsed result is the documented 4-key contract: `call`, `confidence`, `indicators`, `regime`. (The full MCP response also carries `price`, `reasoning`, `timestamp`, and an `_algovault` provenance block if your agent needs them.)

In [3]:
import ast
import json


def _extract_payload(raw):
    """crewai-tools returns tool output as a string.

    AlgoVault replies with two content blocks (a JSON body plus a
    human-readable summary line), so the adapter hands back the string
    repr of a Python list. Recover the first block that parses as JSON.
    """
    if not isinstance(raw, str):
        return raw
    text = raw.strip()
    if text.startswith("["):
        for item in ast.literal_eval(text):
            try:
                return json.loads(item)
            except (ValueError, TypeError):
                continue
    return json.loads(text)


def fetch_trade_call(coin: str, timeframe: str = "4h") -> dict:
    with MCPServerAdapter(server_params) as tools:
        get_trade_call = {t.name: t for t in tools}["get_trade_call"]
        raw = get_trade_call.run(
            coin=coin,
            timeframe=timeframe,
            # Pass concrete values for optional fields CrewAI would
            # otherwise forward as null (see the note below).
            includeReasoning=False,
            exchange="BINANCE",
            assetClass="perp",
        )
    payload = _extract_payload(raw)
    return {k: payload[k] for k in ("call", "confidence", "indicators", "regime")}


result = fetch_trade_call("BTC", "4h")
print(json.dumps(result, indent=2))

# The verdict is one of BUY / SELL / HOLD.
print("Verdict for this run:")
print(result["call"], end="")

{
  "call": "HOLD",
  "confidence": 55,
  "indicators": {
    "funding_rate": 3.998e-05,
    "funding_24h_avg": 3.998e-05,
    "funding_state": "NORMAL",
    "oi_change_pct": 5.39,
    "oi_change_window": "24h",
    "volume_24h": 13089599467.55,
    "trend_persistence": "HIGH",
    "breakout_pending": "IMMINENT",
    "underlying_session": "ALWAYS_OPEN"
  },
  "regime": "TRENDING_DOWN"
}
Verdict for this run:
HOLD

### Note: pass concrete values for optional arguments

AlgoVault's `get_trade_call` declares several optional arguments (`includeReasoning`, `exchange`, `assetClass`). CrewAI's adapter forwards any argument you omit as an explicit `null`, and the server rejects `null` for these fields (it expects the value to be absent, not null). Passing concrete values — `includeReasoning=False`, `exchange="BINANCE"`, `assetClass="perp"` — sidesteps the mismatch.

The two-content-block response is why `_extract_payload` above unpacks a list before parsing JSON: AlgoVault returns the machine-readable JSON and a one-line human summary, and the adapter serializes both.

## 3. (Optional) Wire the tools into a Crew

To let an LLM decide when to call which tool, pass the loaded tools to a CrewAI `Agent` and run a `Crew`. This cell needs an LLM provider key, so it is guarded — it is skipped when `OPENAI_API_KEY` is not set. The direct call above is the key-free path.

In [4]:
import os

if os.environ.get("OPENAI_API_KEY"):
    from crewai import Agent, Crew, Task

    with MCPServerAdapter(server_params) as tools:
        analyst = Agent(
            role="Crypto market analyst",
            goal="Read AlgoVault's composite verdict for the requested asset and explain it.",
            backstory="A quantitative analyst that calls AlgoVault MCP for cross-venue signal.",
            tools=list(tools),
            verbose=True,
        )
        task = Task(
            description="Get a trade call for BTC on the 4h timeframe and summarise the verdict.",
            expected_output="A one-paragraph summary citing call, confidence, and regime.",
            agent=analyst,
        )
        crew = Crew(agents=[analyst], tasks=[task])
        print(crew.kickoff())
else:
    print("Set OPENAI_API_KEY (or another provider key) to run the Crew example.")
    print("The direct get_trade_call above runs without any LLM key.")

Set OPENAI_API_KEY (or another provider key) to run the Crew example.
The direct get_trade_call above runs without any LLM key.


## The four AlgoVault tools

| Tool | Returns |
|---|---|
| `get_trade_call` | Composite BUY/SELL/HOLD verdict + confidence + regime + indicators for one coin and timeframe |
| `get_trade_signal` | Back-compatible alias of `get_trade_call` (same payload) |
| `scan_funding_arb` | Ranked funding-spread opportunities across the 5 venues |
| `get_market_regime` | Regime classification (`TRENDING_UP` / `TRENDING_DOWN` / `RANGING` / `VOLATILE`) for one coin and timeframe |

## Further reading

- Full CrewAI tutorial: [`algovault-skills/docs/integrations/crewai.md`](https://github.com/AlgoVaultLabs/algovault-skills/blob/main/docs/integrations/crewai.md)
- Verify the live track record: [algovault.com/track-record](https://algovault.com/track-record)
- MCP endpoint: `https://api.algovault.com/mcp`